<a href="https://colab.research.google.com/github/dgaida/wpf_cps_th_public/blob/main/assets/exercises/week_02/02_farbraeume_rgb_hsv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Woche 2: Farbräume (RGB & HSV) und Bildausschnitte mit OpenCV

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/wpf_cps_th_public/blob/main/assets/exercises/week_02/02_farbraeume_rgb_hsv.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/dgaida/wpf_cps_th_public/blob/main/assets/exercises/week_02/02_farbraeume_rgb_hsv.ipynb)

In diesem Notebook lernen Sie die Grundlagen der digitalen Bildverarbeitung mit **OpenCV** kennen:
1. **RGB-Farbraum**: Laden eines Bildes, Zerlegung in Rote, Grüne und Blaue Farbkanäle sowie Auswählen eines Bildausschnitts (Region of Interest, ROI).
2. **HSV-Farbraum**: Konvertierung von RGB nach HSV und Analyse der Farbton- (Hue), Sättigungs- (Saturation) und Helligkeitskanäle (Value).

## 1. RGB-Farbraum, Kanalzerlegung & Bildausschnitt (ROI)

Im RGB-Farbraum wird jeder Pixel durch drei Intensitätswerte beschrieben: **Rot (R)**, **Grün (G)** und **Blau (B)** (Wertebereich typischerweise $0$ bis $255$).

*Hinweis:* OpenCV lädt Bilder standardmäßig im **BGR-Format**. Daher müssen wir die Kanäle vor der Darstellung mit Matplotlib nach **RGB** konvertieren (`cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)`).

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
import requests
from io import BytesIO
from PIL import Image

# Verzeichnis erstellen
os.makedirs("data", exist_ok=True)
sample_path = "data/sample_rgb.jpg"

# Beispielbild herunterladen, falls nicht vorhanden
if not os.path.exists(sample_path):
    sample_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e0/Synthese%2FFarben.svg/640px-Synthese%2FFarben.svg.png"
    response = requests.get(sample_url)
    img_pil = Image.open(BytesIO(response.content)).convert("RGB")
    img_pil.save(sample_path)

# Option für eigenes Bild in Colab
use_custom_upload = False  # Setzen Sie auf True, um in Colab ein eigenes Bild hochzuladen

if use_custom_upload:
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            image_file = fname
            break
    except ImportError:
        print("Google Colab nicht aktiv. Verwende Beispielbild.")
        image_file = sample_path
else:
    image_file = sample_path

# Bild mit OpenCV laden (BGR)
img_bgr = cv2.imread(image_file)
if img_bgr is None:
    raise FileNotFoundError(f"Konnte Bild nicht laden: {image_file}")

# Von BGR nach RGB konvertieren
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

h_img, w_img, c_img = img_rgb.shape
print(f"Bild erfolgreich geladen: Auflösung = {w_img} x {h_img} Pixel, Kanäle = {c_img}")

### 1.1 Einzelne RGB-Kanäle darstellen

Wir trennen das RGB-Bild in seine drei Kanäle auf:
- **Roter Kanal**: `img_rgb[:, :, 0]`
- **Grüner Kanal**: `img_rgb[:, :, 1]`
- **Blauer Kanal**: `img_rgb[:, :, 2]`

In [ ]:
r_channel = img_rgb[:, :, 0]
g_channel = img_rgb[:, :, 1]
b_channel = img_rgb[:, :, 2]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img_rgb)
axes[0].set_title("Original RGB-Bild", fontsize=12)
axes[0].axis("off")

axes[1].imshow(r_channel, cmap="Reds")
axes[1].set_title("Roter Kanal (R)", fontsize=12)
axes[1].axis("off")

axes[2].imshow(g_channel, cmap="Greens")
axes[2].set_title("Grüner Kanal (G)", fontsize=12)
axes[2].axis("off")

axes[3].imshow(b_channel, cmap="Blues")
axes[3].set_title("Blauer Kanal (B)", fontsize=12)
axes[3].axis("off")

plt.tight_layout()
plt.show()

### 1.2 Bildausschnitt (ROI) auswählen

Sie können einen Teilbereich des RGB-Bildes auswählen, indem Sie die Startkoordinaten $(x, y)$ sowie die Breite $w$ und Höhe $h$ des Ausschnitts angeben.

In [ ]:
# Parameter für den Bildausschnitt definieren
x = int(w_img * 0.25)  # X-Startpixel (Spalte)
y = int(h_img * 0.25)  # Y-Startpixel (Zeile)
w = int(w_img * 0.50)  # Breite des Ausschnitts
h = int(h_img * 0.50)  # Höhe des Ausschnitts

print(f"Ausschnitt definiert von Pixel (x={x}, y={y}) mit Breite w={w} und Höhe h={h}")

# Bildausschnitt per Slicing extrahieren: img[y : y+h, x : x+w]
roi_rgb = img_rgb[y : y + h, x : x + w]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Bild mit Rahmen um ROI
img_with_rect = img_rgb.copy()
cv2.rectangle(img_with_rect, (x, y), (x + w, y + h), (255, 255, 0), 3)
axes[0].imshow(img_with_rect)
axes[0].set_title("Originalbild mit ROI-Rahmen", fontsize=12)
axes[0].axis("off")

# Ausgewählter Ausschnitt
axes[1].imshow(roi_rgb)
axes[1].set_title(f"Bildausschnitt (ROI: {w}x{h} px)", fontsize=12)
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 2. HSV-Farbraum (Hue, Saturation, Value)

Der **HSV-Farbraum** beschreibt Farben menschlicher und intuitiver als RGB:
- **Hue (H - Farbton)**: Der eigentliche Farbton (z. B. Rot, Gelb, Grün, Blau) als Winkel auf einem Farbkreis. In OpenCV wird der Bereich $0^\circ$ bis $360^\circ$ auf $0$ bis $179$ skaliert.
- **Saturation (S - Farbsättigung)**: Wie 'rein' oder 'graustichig' die Farbe ist ($0$ bis $255$).
- **Value (V - Helligkeit)**: Die Helligkeit der Farbe ($0$ = Schwarz, $255$ = volle Helligkeit).

In der Bildverarbeitung eignet sich HSV besonders gut für die robuste **Farbbasierte Objekterkennung und Segmentierung**, da Farbton (H) unabhängig von Beleuchtungsstärke (V) gefiltered werden kann.

In [ ]:
# Konvertierung von RGB nach HSV mit OpenCV
img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)

h_channel = img_hsv[:, :, 0]  # Hue (Farbton: 0..179)
s_channel = img_hsv[:, :, 1]  # Saturation (Sättigung: 0..255)
v_channel = img_hsv[:, :, 2]  # Value (Helligkeit: 0..255)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(img_rgb)
axes[0].set_title("Original RGB-Bild", fontsize=12)
axes[0].axis("off")

im_h = axes[1].imshow(h_channel, cmap="hsv")
axes[1].set_title("Hue Kanal (H - Farbton)", fontsize=12)
axes[1].axis("off")
fig.colorbar(im_h, ax=axes[1], fraction=0.046, pad=0.04)

im_s = axes[2].imshow(s_channel, cmap="gray")
axes[2].set_title("Saturation Kanal (S - Sättigung)", fontsize=12)
axes[2].axis("off")
fig.colorbar(im_s, ax=axes[2], fraction=0.046, pad=0.04)

im_v = axes[3].imshow(v_channel, cmap="gray")
axes[3].set_title("Value Kanal (V - Helligkeit)", fontsize=12)
axes[3].axis("off")
fig.colorbar(im_v, ax=axes[3], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 3. Zusammenfassung

- **RGB**: Das Bild besteht aus drei Farbkanälen (Rot, Grün, Blau). OpenCV liest Bilder standardmäßig im **BGR-Format** ein.
- **ROI (Region of Interest)**: Mit NumPy-Slicing (`img[y : y+h, x : x+w]`) lassen sich Bildbereiche anhand von Pixelkoordinaten, Breite und Höhe extrahieren.
- **HSV**: Trennt den **Farbton (Hue)** von **Sättigung (Saturation)** und **Helligkeit (Value)**, was die Erkennung von Objekten bei wechselnden Lichtverhältnissen erleichtert.